# Zomato Restaurant Recommendation System

Content-based recommendations using **TF-IDF** and **Cosine Similarity**.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.preprocess import load_dataset
from src.recommender import ZomatoRecommender
from src.evaluation import evaluate_recommender
from src.ab_testing import run_ab_test

DATA_PATH = ROOT / "data" / "sample_zomato.csv"
if not DATA_PATH.exists():
    from scripts.generate_sample_data import generate_sample_dataset
    generate_sample_dataset().to_csv(DATA_PATH, index=False)

df = load_dataset(DATA_PATH)
print(f"Loaded {len(df)} restaurants")
df.head()

In [ ]:
model = ZomatoRecommender(max_features=5000, ngram_range=(1, 2))
model.fit(df)

sample = df.iloc[0]["Restaurant Name"]
print(f"Restaurants similar to: {sample}\n")

for i, rec in enumerate(model.recommend_by_restaurant(sample, top_n=5), 1):
    print(f"{i}. {rec.restaurant_name} | {rec.city} | {rec.rating:.1f} | score={rec.similarity_score:.3f}")
    print(f"   {rec.cuisines}")

In [ ]:
prefs = model.recommend_by_preferences(
    cuisines="North Indian, Mughlai",
    city="New Delhi",
    price_range=2,
    top_n=5,
)

for i, rec in enumerate(prefs, 1):
    print(f"{i}. {rec.restaurant_name} | {rec.locality} | range {rec.price_range} | {rec.rating:.1f}")

In [ ]:
metrics = evaluate_recommender(model, k=10)
print(f"Precision@10: {metrics.precision_at_k:.3f}")
print(f"Recall@10:    {metrics.recall_at_k:.3f}")
print(f"Hit Rate:       {metrics.hit_rate:.3f}")
print(f"MAP:            {metrics.mean_average_precision:.3f}")

print("\n--- A/B Test ---")
ab = run_ab_test(df, sample_size=100)
for v in ab.variants:
    print(f"{v.name}: satisfaction={v.user_satisfaction_score:.3f}, precision@10={v.metrics.precision_at_k:.3f}")
print(f"Winner: {ab.winner} (+{ab.improvement_pct}%)")